# 01 · Automated Data Downloader

Reads `config/data_sources.csv` and downloads all files to the correct folders.

**To add or change a source:** edit `config/data_sources.csv` only. Do not change this notebook.

### Folder structure created:
```
data/
├── raw/
│   ├── boe/      ← Bank of England files
│   ├── ons/      ← ONS Economy files
│   ├── hpi/      ← UK House Price Index files
│   └── fred/     ← FRED macro files
├── processed/    ← cleaned/merged outputs (later notebooks)
└── outputs/      ← dashboard and reports (later notebooks)
```

In [1]:
import pandas as pd
import requests
import time
from pathlib import Path

# ── Config ────────────────────────────────────────────────────────────────
CONFIG_FILE  = 'data/config/data_sources.csv'
DATA_ROOT    = Path('data')
DELAY_S      = 2       # polite delay between requests (seconds)
TIMEOUT_S    = 60      # per-file download timeout
OVERWRITE    = False   # set True to re-download files that already exist

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (compatible; DataAnalytics/1.0; public data research)',
    'Accept': 'text/csv, text/plain, application/octet-stream, */*',
}
# ──────────────────────────────────────────────────────────────────────────

sources = pd.read_csv(CONFIG_FILE)
print(f'Loaded {len(sources)} sources from {CONFIG_FILE}')
sources[['source', 'series_label', 'folder', 'filename']]

Loaded 12 sources from data/config/data_sources.csv


,source,series_label,folder,filename
0,Bank of England,base_rate,raw/boe,boe_base_rate.csv
1,Bank of England,mortgage_approvals,raw/boe,boe_mortgage_approvals.csv
2,Bank of England,consumer_credit,raw/boe,boe_consumer_credit.csv
3,Bank of England,sme_lending,raw/boe,boe_sme_lending.csv
4,ONS,cpi_inflation,raw/ons,ons_cpi_inflation.csv
5,ONS,gdp_growth,raw/ons,ons_gdp_growth.csv
6,ONS,unemployment_rate,raw/ons,ons_unemployment.csv
7,ONS,avg_weekly_earnings,raw/ons,ons_avg_weekly_earnings.csv
8,UK HPI,full_dataset,raw/hpi,hpi_full_2024_12.csv
9,FRED,gilt_10yr,raw/fred,fred_gilt_10yr.csv


In [2]:
def create_folders(sources: pd.DataFrame, data_root: Path) -> None:
    """Create all required subfolders plus processed/ and outputs/."""
    for folder in sources['folder'].unique():
        (data_root / folder).mkdir(parents=True, exist_ok=True)
    (data_root / 'processed').mkdir(parents=True, exist_ok=True)
    (data_root / 'outputs').mkdir(parents=True, exist_ok=True)
    print('Folders ready:')
    for p in sorted(data_root.rglob('*')):
        if p.is_dir():
            print(f'  {p}')

create_folders(sources, DATA_ROOT)

Folders ready:
  data/config
  data/logs
  data/outputs
  data/processed
  data/raw
  data/raw/boe
  data/raw/fred
  data/raw/hpi
  data/raw/ons


In [3]:
def download_file(url: str, dest: Path, timeout: int, headers: dict) -> dict:
    """Download a single file. Returns a status dict."""
    try:
        response = requests.get(url, headers=headers, timeout=timeout)
        response.raise_for_status()
        dest.write_bytes(response.content)
        size_kb = len(response.content) / 1024
        return {'status': 'OK', 'size_kb': round(size_kb, 1), 'error': None}
    except requests.HTTPError as e:
        return {'status': 'HTTP_ERROR', 'size_kb': 0, 'error': str(e)}
    except requests.Timeout:
        return {'status': 'TIMEOUT', 'size_kb': 0, 'error': f'Timed out after {timeout}s'}
    except Exception as e:
        return {'status': 'FAILED', 'size_kb': 0, 'error': str(e)}


results = []

for _, row in sources.iterrows():
    dest = DATA_ROOT / row['folder'] / row['filename']

    # Skip if file exists and overwrite is off
    if dest.exists() and not OVERWRITE:
        size_kb = round(dest.stat().st_size / 1024, 1)
        print(f'  SKIP  {row["series_label"]:30s} (already exists, {size_kb} KB)')
        results.append({**row.to_dict(), 'status': 'SKIPPED', 'size_kb': size_kb, 'error': None, 'path': str(dest)})
        continue

    print(f'  GET   {row["series_label"]:30s} → {dest.name}', end=' ... ')
    result = download_file(row['url'], dest, TIMEOUT_S, HEADERS)
    result['path'] = str(dest)
    results.append({**row.to_dict(), **result})

    if result['status'] == 'OK':
        print(f"✅  {result['size_kb']} KB")
    else:
        print(f"❌  {result['status']}: {result['error']}")

    time.sleep(DELAY_S)

  SKIP  base_rate                      (already exists, 5.1 KB)
  SKIP  mortgage_approvals             (already exists, 1.7 KB)
  SKIP  consumer_credit                (already exists, 5.6 KB)
  SKIP  sme_lending                    (already exists, 18.6 KB)
  SKIP  cpi_inflation                  (already exists, 10.4 KB)
  SKIP  gdp_growth                     (already exists, 4.7 KB)
  SKIP  unemployment_rate              (already exists, 15.4 KB)
  SKIP  avg_weekly_earnings            (already exists, 7.4 KB)
  SKIP  full_dataset                   (already exists, 32121.0 KB)
  SKIP  gilt_10yr                      (already exists, 14.2 KB)
  SKIP  gbp_usd                        (already exists, 249.7 KB)
  SKIP  uk_cpi_oecd                    (already exists, 25.5 KB)


In [4]:
# ── Download Summary ──────────────────────────────────────────────────────
summary = pd.DataFrame(results)[['source', 'series_label', 'filename', 'status', 'size_kb', 'error']]

ok      = (summary['status'] == 'OK').sum()
skipped = (summary['status'] == 'SKIPPED').sum()
failed  = summary[~summary['status'].isin(['OK', 'SKIPPED'])]

print(f'\n── Download Summary ──────────────────────')
print(f'  ✅  Downloaded : {ok}')
print(f'  ⏭️   Skipped   : {skipped}  (set OVERWRITE=True to re-download)')
print(f'  ❌  Failed     : {len(failed)}')

if not failed.empty:
    print('\nFailed downloads — check URLs in config/data_sources.csv:')
    display(failed[['series_label', 'status', 'error']])

print()
summary.style.applymap(
    lambda v: 'background-color:#d1e7dd' if v=='OK'
    else ('background-color:#fff3cd' if v=='SKIPPED'
    else 'background-color:#f8d7da'),
    subset=['status']
)


── Download Summary ──────────────────────
  ✅  Downloaded : 0
  ⏭️   Skipped   : 12  (set OVERWRITE=True to re-download)
  ❌  Failed     : 0



/tmp/ipykernel_1271452/377510.py:18: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  summary.style.applymap(


,source,series_label,filename,status,size_kb,error
0,Bank of England,base_rate,boe_base_rate.csv,SKIPPED,5.100000,None
1,Bank of England,mortgage_approvals,boe_mortgage_approvals.csv,SKIPPED,1.700000,None
2,Bank of England,consumer_credit,boe_consumer_credit.csv,SKIPPED,5.600000,None
3,Bank of England,sme_lending,boe_sme_lending.csv,SKIPPED,18.600000,None
4,ONS,cpi_inflation,ons_cpi_inflation.csv,SKIPPED,10.400000,None
5,ONS,gdp_growth,ons_gdp_growth.csv,SKIPPED,4.700000,None
6,ONS,unemployment_rate,ons_unemployment.csv,SKIPPED,15.400000,None
7,ONS,avg_weekly_earnings,ons_avg_weekly_earnings.csv,SKIPPED,7.400000,None
8,UK HPI,full_dataset,hpi_full_2024_12.csv,SKIPPED,32121.000000,None
9,FRED,gilt_10yr,fred_gilt_10yr.csv,SKIPPED,14.200000,None


In [5]:
# ── Save download log ─────────────────────────────────────────────────────
# Useful audit trail — records what was downloaded and when
from datetime import datetime

log = pd.DataFrame(results)[['source', 'series_label', 'filename', 'folder', 'status', 'size_kb', 'error', 'path']]
log['downloaded_at'] = datetime.now().strftime('%Y-%m-%d %H:%M')
log_path = DATA_ROOT / 'processed' / 'download_log.csv'
log.to_csv(log_path, index=False)
print(f'Download log saved → {log_path}')

Download log saved → data/processed/download_log.csv
